# Design Smell Detector: fine-tune a small LM with Unsloth (T4, ~1-1.5 h)

Detects 6 design smells in Mermaid UML class diagrams: **God Class, Cyclic Dependency, Hub-like Modularization, Deep Hierarchy, Wide Hierarchy, Data Class**.

**Setup:** Runtime > Change runtime type > **T4 GPU**, then Run all.

The dataset is generated by rules (cell 3), so labels are consistent. Smells are injected into random diagrams, near-threshold "hard negatives" are mixed in, and labels come from the detector functions, not by hand. Train, val, and test sets never share a diagram.

| Smell | Rule (ground truth) | Severity |
|---|---|---|
| God Class | >=12 methods AND >=4 outgoing deps | 12-13 low, 14-16 medium, 17+ high |
| Cyclic Dependency | class is in a dependency cycle (`-->`/`..>`); every member is reported | 2 classes low, 3 medium, 4+ high |
| Hub-like Modularization | >=5 incoming AND >=5 outgoing deps | min(in,out) 5 low, 6-7 medium, 8+ high |
| Deep Hierarchy | leaf class with inheritance depth >=4 | 4 low, 5-6 medium, 7+ high |
| Wide Hierarchy | parent with >=8 direct subclasses | 8-9 low, 10-12 medium, 13+ high |
| Data Class | >=6 attributes and 0 methods | 6-7 low, 8-9 medium, 10+ high |

These are heuristic definitions (inspired by common smell metrics)

In [1]:
!nvidia-smi

Thu Nov  5 18:28:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%capture
!pip install unsloth

## 3. Dataset generator (rule-based labels)

In [3]:
import random, json, hashlib
from collections import Counter, defaultdict

SMELLS = ["God Class", "Cyclic Dependency", "Hub-like Modularization",
          "Deep Hierarchy", "Wide Hierarchy", "Data Class"]
SYSTEM = ("You detect design smells in UML class diagrams written in Mermaid. "
          "Possible smells: God Class, Cyclic Dependency, Hub-like Modularization, "
          "Deep Hierarchy, Wide Hierarchy, Data Class. "
          "Output only a JSON array of objects with keys 'smell', 'class', 'severity' "
          "(high/medium/low). Output [] if there are no smells.")

NOUNS = ["Order","Customer","Product","Invoice","Payment","Shipment","User","Account","Cart","Ticket",
         "Booking","Course","Student","Patient","Appointment","Employee","Vehicle","Reservation","Review",
         "Coupon","Warehouse","Supplier","Report","Session","Notification","Subscription","Playlist",
         "Document","Project","Task","Invoice","Device","Event","Message","Profile","Catalog","Loan"]
ROLES = ["Service","Repository","Controller","Manager","Handler","Factory","Validator","Mapper",
         "Gateway","Processor","Client","Adapter","Provider","Builder","Store","Helper"]
VERBS = ["create","update","delete","find","load","save","validate","send","calculate","render","export",
         "import","sync","archive","notify","schedule","audit","cancel","approve","reject","merge","parse",
         "format","encrypt","fetch","publish","refresh","generate","check","apply","track","resolve"]
ATTRS = ["id","name","email","status","createdAt","updatedAt","price","quantity","address","phone","type",
         "title","description","owner","total","currency","version","code","label","priority","country",
         "city","zip","notes","rating","weight","color","size","url","token"]
TYPES = ["String","int","Date","double","boolean","long"]

class D:
    def __init__(s, rng):
        s.r = rng
        s.style = rng.choices(["domain","letters","module"], [0.6,0.2,0.2])[0]
        s.cls = {}; s.deps = set(); s.inh = set(); s.dotted = set()
        s.counter = 0
    def name(s):
        r = s.r
        while True:
            if s.style == "domain":
                n = r.choice(NOUNS) + (r.choice(ROLES) if r.random() < 0.6 else "")
            elif s.style == "letters":
                n = chr(65 + s.counter) if s.counter < 26 else f"C{s.counter}"
            else:
                n = f"Module{s.counter+1}"
            s.counter += 1
            if n not in s.cls: return n
            if s.style == "domain" and s.counter > 200: return f"{n}{s.counter}"
    def add(s, attrs=None, ops=None):
        r = s.r
        n = s.name()
        attrs = r.randint(0, 3) if attrs is None else attrs
        ops = r.randint(0, 5) if ops is None else ops
        a = []
        pool = ATTRS[:]; r.shuffle(pool)
        for i in range(attrs):
            base = pool[i] if i < len(pool) else f"{pool[i%len(pool)]}{i}"
            vis = r.choice("+-")
            a.append(f"{vis}{r.choice(TYPES)+' ' if r.random()<0.4 else ''}{base}")
        o = []
        used = set()
        while len(o) < ops:
            m = r.choice(VERBS) + r.choice(NOUNS)
            if m in used: continue
            used.add(m)
            o.append(f"+{m}(){' '+r.choice(TYPES) if r.random()<0.25 else ''}")
        s.cls[n] = {"a": a, "o": o}
        return n
    def dep(s, a, b):
        if a != b: s.deps.add((a, b))
    def render(s):
        r = s.r
        lines = ["classDiagram"]
        names = list(s.cls); r.shuffle(names)
        for n in names:
            c = s.cls[n]
            if c["a"] or c["o"]:
                lines.append(f"  class {n} {{")
                for m in c["a"] + c["o"]: lines.append(f"    {m}")
                lines.append("  }")
            else:
                lines.append(f"  class {n}")
        edges = [f"  {a} {'..>' if (a,b) in s.dotted else '-->'} {b}" for a, b in s.deps]
        edges += [f"  {c} --|> {p}" for c, p in s.inh]
        r.shuffle(edges)
        return "\n".join(lines + edges)

# ---------------- detectors (ground truth rules) ----------------
def sev(v, lo, hi): return "low" if v < lo else ("medium" if v < hi else "high")

def analyze(d):
    out = []
    outd = Counter(a for a, b in d.deps); ind = Counter(b for a, b in d.deps)
    for n, c in d.cls.items():
        if len(c["o"]) >= 12 and outd[n] >= 4:                      # God Class
            out.append(("God Class", n, sev(len(c["o"]), 14, 17)))
        if ind[n] >= 5 and outd[n] >= 5:                            # Hub-like
            out.append(("Hub-like Modularization", n, sev(min(ind[n], outd[n]), 6, 8)))
        if len(c["a"]) >= 6 and len(c["o"]) == 0:                   # Data Class
            out.append(("Data Class", n, sev(len(c["a"]), 8, 10)))
    # cycles via Tarjan SCC
    g = defaultdict(list)
    for a, b in d.deps: g[a].append(b)
    idx = {}; low = {}; st = []; on = set(); sccs = []; i = [0]
    def sc(v):
        idx[v] = low[v] = i[0]; i[0] += 1; st.append(v); on.add(v)
        for w in g[v]:
            if w not in idx: sc(w); low[v] = min(low[v], low[w])
            elif w in on: low[v] = min(low[v], idx[w])
        if low[v] == idx[v]:
            comp = []
            while True:
                w = st.pop(); on.discard(w); comp.append(w)
                if w == v: break
            sccs.append(comp)
    for v in list(d.cls):
        if v not in idx: sc(v)
    for comp in sccs:
        if len(comp) >= 2:
            sv = "low" if len(comp) == 2 else ("medium" if len(comp) == 3 else "high")
            for n in comp: out.append(("Cyclic Dependency", n, sv))
    # hierarchy
    parent = {c: p for c, p in d.inh}
    kids = Counter(p for c, p in d.inh)
    def depth(n):
        k = 0
        while n in parent: n = parent[n]; k += 1
        return k
    for n in d.cls:
        if n not in kids and depth(n) >= 4:                         # Deep Hierarchy (leaf)
            out.append(("Deep Hierarchy", n, sev(depth(n), 5, 7)))
        if kids[n] >= 8:                                            # Wide Hierarchy
            out.append(("Wide Hierarchy", n, sev(kids[n], 10, 13)))
    out.sort(key=lambda x: (SMELLS.index(x[0]), x[1]))
    return [{"smell": s, "class": c, "severity": v} for s, c, v in out]

# ---------------- background (clean) ----------------
def background(d):
    r = d.r
    n = r.randint(2, 12)
    ns = [d.add(ops=r.randint(0, 6), attrs=r.randint(0, 4)) for _ in range(n)]
    outd = Counter(); ind = Counter()
    for _ in range(r.randint(0, int(n * 1.5))):
        i, j = sorted(r.sample(range(n), 2)) if n > 1 else (0, 0)
        a, b = ns[i], ns[j]
        if a != b and outd[a] < 3 and ind[b] < 4 and (a, b) not in d.deps:
            d.dep(a, b); outd[a] += 1; ind[b] += 1
            if r.random() < 0.2: d.dotted.add((a, b))
    if r.random() < 0.5:
        p = d.add(); ks = [d.add(ops=r.randint(0, 3)) for _ in range(r.randint(2, 5))]
        for k in ks: d.inh.add((k, p))
        if r.random() < 0.3:
            c = d.add(); d.inh.add((c, ks[0]))

# ---------------- smell injectors ----------------
def pick_targets(d, k, exclude):
    r = d.r
    pool = [n for n in d.cls if n not in exclude]
    r.shuffle(pool)
    t = pool[:k]
    while len(t) < k: t.append(d.add(ops=r.randint(0, 3)))
    return t

def god(d, ops=None, outdeg=None):
    r = d.r
    g = d.add(attrs=r.randint(0, 3), ops=ops or r.randint(12, 18))
    for t in pick_targets(d, outdeg or r.randint(4, 7), {g}): d.dep(g, t)

def cycle(d, k=None):
    r = d.r
    k = k or r.choice([2, 2, 3, 3, 4, 5])
    ring = [d.add(ops=r.randint(0, 4)) for _ in range(k)]
    for i in range(k): d.dep(ring[i], ring[(i + 1) % k])
    if r.random() < 0.4 and k >= 3:   # extra chord, still one SCC
        d.dep(ring[0], ring[2])
    others = [n for n in d.cls if n not in ring]
    if others and r.random() < 0.6:
        o = r.choice(others)
        if r.random() < 0.5: d.dep(o, r.choice(ring))
        elif not has_path(d, o, set(ring)): d.dep(r.choice(ring), o)

def has_path(d, src, targets):
    g = defaultdict(list)
    for a, b in d.deps: g[a].append(b)
    seen = set(); stack = [src]
    while stack:
        v = stack.pop()
        if v in targets: return True
        if v in seen: continue
        seen.add(v); stack += g[v]
    return False

def hub(d, nin=None, nout=None):
    r = d.r
    h = d.add(ops=r.randint(1, 6))
    nin = r.randint(5, 9) if nin is None else nin
    nout = r.randint(5, 9) if nout is None else nout
    srcs = {d.add(ops=r.randint(0, 3)) for _ in range(nin)}      # fresh sources: no cycles
    for x in srcs: d.dep(x, h)
    for t in pick_targets(d, nout, {h} | srcs):
        if not has_path(d, t, {h}): d.dep(h, t)

def deep(d, depth=None):
    r = d.r
    depth = depth or r.choice([4, 4, 5, 5, 6, 7, 8])
    prev = d.add(ops=r.randint(1, 3))
    for _ in range(depth):
        c = d.add(ops=r.randint(0, 3)); d.inh.add((c, prev)); prev = c

def wide(d, k=None):
    r = d.r
    k = k or r.choice([8, 9, 10, 11, 12, 13, 14])
    p = d.add(ops=r.randint(1, 4))
    for _ in range(k): d.inh.add((d.add(attrs=r.randint(0, 2), ops=r.randint(0, 2)), p))

def data(d, attrs=None):
    r = d.r
    c = d.add(attrs=attrs or r.randint(6, 11), ops=0)
    others = [n for n in d.cls if n != c]
    for o in r.sample(others, min(len(others), r.randint(0, 2))): d.dep(o, c)

def diamond(d):
    a, b, c, e = [d.add(ops=d.r.randint(0, 3)) for _ in range(4)]
    for x, y in [(a, b), (a, c), (b, e), (c, e)]: d.dep(x, y)

INJECT = {"God Class": god, "Cyclic Dependency": cycle, "Hub-like Modularization": hub,
          "Deep Hierarchy": deep, "Wide Hierarchy": wide, "Data Class": data}

def near_miss(d, kind):
    r = d.r
    if kind == "God Class":
        if r.random() < 0.5: god(d, ops=r.randint(8, 11), outdeg=r.randint(4, 7))
        else: god(d, ops=r.randint(12, 17), outdeg=r.randint(0, 3))
    elif kind == "Hub-like Modularization":
        if r.random() < 0.5: hub(d, nin=r.randint(5, 9), nout=r.randint(0, 4))
        else: hub(d, nin=r.randint(0, 4), nout=r.randint(5, 9))
    elif kind == "Deep Hierarchy": deep(d, depth=r.choice([2, 3]))
    elif kind == "Wide Hierarchy": wide(d, k=r.randint(5, 7))
    elif kind == "Data Class":
        if r.random() < 0.5: data(d, attrs=r.randint(4, 5))
        else:
            c = d.add(attrs=r.randint(6, 10), ops=r.randint(2, 5))
    elif kind == "Cyclic Dependency": diamond(d)

def make_example(rng):
    d = D(rng)
    background(d)
    k = rng.choices([0, 1, 2, 3], [0.28, 0.45, 0.22, 0.05])[0]
    chosen = rng.sample(SMELLS, k)
    for s in chosen: INJECT[s](d)
    rest = [s for s in SMELLS if s not in chosen]
    for s in rng.sample(rest, rng.choice([0, 1, 1, 2])): near_miss(d, s)
    if len(d.cls) > 45: return None
    return d.render(), analyze(d)

def build(n_train=3500, n_val=150, n_test=400, seed=42):
    rng = random.Random(seed); seen = set(); data_ = []
    need = n_train + n_val + n_test
    while len(data_) < need:
        ex = make_example(rng)
        if not ex: continue
        h = hashlib.md5(ex[0].encode()).hexdigest()
        if h in seen: continue
        seen.add(h); data_.append(ex)
    def fmt(ex):
        return {"messages": [{"role": "system", "content": SYSTEM},
                             {"role": "user", "content": ex[0]},
                             {"role": "assistant", "content": json.dumps(ex[1])}]}
    rows = [fmt(e) for e in data_]
    return rows[:n_train], rows[n_train:n_train+n_val], rows[n_train+n_val:]

## 4. Build train / val / test and inspect

In [4]:
import json, collections
train_rows, val_rows, test_rows = build(n_train=3500, n_val=150, n_test=400, seed=42)

for name, rows in [("design_smell_train", train_rows), ("design_smell_val", val_rows), ("design_smell_test", test_rows)]:
    with open(f"{name}.jsonl", "w") as f:
        for r in rows: f.write(json.dumps(r) + "\n")

cnt = collections.Counter(); clean = 0
for r in train_rows:
    lab = json.loads(r["messages"][2]["content"])
    clean += (not lab)
    for s in {x["smell"] for x in lab}: cnt[s] += 1
print(f"train={len(train_rows)} val={len(val_rows)} test={len(test_rows)} | clean diagrams in train: {clean}")
print("diagrams containing each smell:", dict(cnt))
print()
print(train_rows[1]["messages"][1]["content"])
print("->", train_rows[1]["messages"][2]["content"])

train=3500 val=150 test=400 | clean diagrams in train: 956
diagrams containing each smell: {'Deep Hierarchy': 627, 'Cyclic Dependency': 591, 'Data Class': 597, 'Wide Hierarchy': 606, 'God Class': 654, 'Hub-like Modularization': 611}

classDiagram
  class Module3 {
    -int color
    +int title
    -token
    +loadTask()
    +syncNotification()
    +scheduleTicket()
    +createUser()
    +notifyCourse()
  }
  class Module8 {
    -label
  }
  class Module15 {
    +parseSubscription()
    +parseProfile()
    +mergeDevice()
  }
  class Module7 {
    +resolveWarehouse() int
    +formatVehicle()
    +cancelCatalog() int
  }
  class Module6 {
    +quantity
    +double code
    -long email
    +parseOrder()
  }
  class Module1 {
    +cancelReport()
    +publishProfile()
    +updateAccount()
  }
  class Module9 {
    +rating
    +priority
    +resolveUser()
  }
  class Module14 {
    -boolean token
    -name
    +mergeStudent()
    +scheduleLoan()
  }
  class Module13 {
    +saveAppointment()
 

## 5. Load a 1.5B model in 4-bit

In [5]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 3072
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",   # try "unsloth/Qwen2.5-3B-Instruct-bnb-4bit" for more accuracy (~2x slower)
    max_seq_length = max_seq_length,
    dtype = None,          # auto: fp16 on T4
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth", random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.2: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.11.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


## 6. Format data with the chat template

In [6]:
from datasets import Dataset

def to_text(rows):
    return Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize=False)} for r in rows])

train_ds = to_text(train_rows)
lens = [len(tokenizer(t)["input_ids"]) for t in train_ds["text"][:300]]
print("token length (first 300): mean", sum(lens)//len(lens), "max", max(lens))

token length (first 300): mean 737 max 1677


## 7. Train (2 epochs, loss only on the answer)

In [7]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = train_ds,
    dataset_text_field = "text", max_seq_length = max_seq_length, packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 8,   # effective batch 16
        num_train_epochs = 2, learning_rate = 2e-4, warmup_steps = 10,
        lr_scheduler_type = "linear", optim = "adamw_8bit", weight_decay = 0.01,
        fp16 = not is_bfloat16_supported(), bf16 = is_bfloat16_supported(),
        logging_steps = 10, save_strategy = "no", report_to = "none",
        seed = 3407, output_dir = "outputs", dataset_num_proc = 2,
    ),
)
trainer = train_on_responses_only(
    trainer, instruction_part = "<|im_start|>user\n", response_part = "<|im_start|>assistant\n")
stats = trainer.train()
print(f"Training time: {stats.metrics['train_runtime']/60:.1f} min")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3500 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,500 | Num Epochs = 2 | Total steps = 438
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,0.368935
20,0.279288
30,0.245533
40,0.221841
50,0.196015
60,0.175939
70,0.178690
80,0.176500
90,0.167009
100,0.160088


Training time: 58.3 min


## 8. Evaluate on the held-out test set

In [8]:
import json, torch
from collections import defaultdict
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"

def predict(users, bs=8, max_new=300):
    order = sorted(range(len(users)), key=lambda i: len(users[i])); res = [None]*len(users)
    for s in range(0, len(order), bs):
        idx = order[s:s+bs]
        prompts = [tokenizer.apply_chat_template(
            [{"role":"system","content":SYSTEM},{"role":"user","content":users[i]}],
            tokenize=False, add_generation_prompt=True) for i in idx]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
        for i, o in zip(idx, tokenizer.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)):
            res[i] = o.strip()
    return res

def parse(s):
    try:
        v = json.loads(s)
        return [(x["smell"], x["class"], x["severity"]) for x in v], True
    except Exception:
        return [], False

users = [r["messages"][1]["content"] for r in test_rows]
gold  = [[(x["smell"], x["class"], x["severity"]) for x in json.loads(r["messages"][2]["content"])] for r in test_rows]
preds = predict(users)

tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int); sev_ok = sev_tot = 0
exact = invalid = 0; clean_ok = clean_tot = 0
for p_raw, g in zip(preds, gold):
    p, ok = parse(p_raw); invalid += (not ok)
    gk = {(s, c): v for s, c, v in g}; pk = {(s, c): v for s, c, v in p}
    exact += (set(gk.items()) == set(pk.items()))
    if not g: clean_tot += 1; clean_ok += (not p)
    for k in pk:
        if k in gk: tp[k[0]] += 1; sev_tot += 1; sev_ok += (pk[k] == gk[k])
        else: fp[k[0]] += 1
    for k in gk:
        if k not in pk: fn[k[0]] += 1

print(f"Invalid JSON: {invalid}/{len(preds)}")
print(f"Exact match (smell+class+severity, whole diagram): {exact/len(preds):.1%}")
print(f"Clean diagrams correctly answered []: {clean_ok}/{clean_tot}")
print(f"Severity accuracy on correctly detected (smell,class): {sev_ok/max(sev_tot,1):.1%}\n")
print(f"{'Smell':26s} {'P':>6s} {'R':>6s} {'F1':>6s}  (detection of smell+class)")
for s in SMELLS:
    P = tp[s]/max(tp[s]+fp[s],1); R = tp[s]/max(tp[s]+fn[s],1); F = 2*P*R/max(P+R,1e-9)
    print(f"{s:26s} {P:6.2f} {R:6.2f} {F:6.2f}")

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Invalid JSON: 0/400
Exact match (smell+class+severity, whole diagram): 35.8%
Clean diagrams correctly answered []: 83/109
Severity accuracy on correctly detected (smell,class): 44.1%

Smell                           P      R     F1  (detection of smell+class)
God Class                    0.87   0.90   0.89
Cyclic Dependency            0.50   0.23   0.31
Hub-like Modularization      0.79   0.90   0.84
Deep Hierarchy               0.52   0.53   0.53
Wide Hierarchy               0.76   0.67   0.71
Data Class                   0.91   0.93   0.92


## 9. Try your own diagram

In [9]:
my_diagram = """classDiagram
  class OrderService {
    +createOrder()
    +cancelOrder()
    +calculateTax()
    +sendEmail()
    +generateReport()
    +validateUser()
    +backupDatabase()
    +renderInvoice()
    +applyDiscount()
    +refundPayment()
    +exportCsv()
    +auditLog()
  }
  class UserRepo
  class EmailService
  class TaxCalculator
  class ReportGenerator
  OrderService --> UserRepo
  OrderService --> EmailService
  OrderService --> TaxCalculator
  OrderService --> ReportGenerator
"""
print(predict([my_diagram])[0])

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[]


## 10. Save the LoRA adapter and download it

In [10]:
model.save_pretrained("design_smell_lora"); tokenizer.save_pretrained("design_smell_lora")
!zip -r design_smell_lora.zip design_smell_lora design_smell_train.jsonl design_smell_val.jsonl design_smell_test.jsonl
from google.colab import files
files.download("design_smell_lora.zip")

Unsloth: Restored added_tokens_decoder metadata in design_smell_lora/tokenizer_config.json.


  adding: design_smell_lora/ (stored 0%)
  adding: design_smell_lora/README.md (deflated 65%)
  adding: design_smell_lora/chat_template.jinja (deflated 71%)
  adding: design_smell_lora/tokenizer.json (deflated 81%)
  adding: design_smell_lora/adapter_config.json (deflated 60%)
  adding: design_smell_lora/adapter_model.safetensors (deflated 7%)
  adding: design_smell_lora/tokenizer_config.json (deflated 89%)
  adding: design_smell_train.jsonl (deflated 85%)
  adding: design_smell_val.jsonl (deflated 85%)
  adding: design_smell_test.jsonl (deflated 85%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>